# rerank on a Colab GPU

Runs the same command the Kaggle kernel runs. Before you start:

1. **Runtime -> Change runtime type -> T4 GPU.**
2. **Secrets** (the key icon in the left sidebar) -> add `OPENROUTER_API_KEY`
   and give this notebook access. It is read at run time and never printed.

Then run every cell. The results and the raw wire log end up in `/content/out`,
and the last cell zips them for download.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!git clone --depth 1 --filter=blob:none --sparse https://github.com/nadeem4/ai-experiments.git /content/ai-experiments
!git -C /content/ai-experiments sparse-checkout set cli rerank
%cd /content/ai-experiments

In [ ]:
!pip install -q "laya>=0.3.20" "rank-bm25>=0.2.2" "pytrec-eval-terrier>=0.5.7"     "sentence-transformers>=5.0" "huggingface-hub>=1.0"

In [ ]:
import os

from google.colab import userdata

# Only jev-score leaves the machine. Without the key it is dropped rather than
# run: a failed call is recorded like any other, so 6,460 of them would be
# skipped as already done by the run that has the key.
METHODS = ["hybrid", "laya-score", "laya-typed-score", "jev-score", "cross-encoder"]
try:
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")
    print("OPENROUTER_API_KEY: loaded")
except Exception as e:
    METHODS = [m for m in METHODS if not m.startswith("jev")]
    print(f"OPENROUTER_API_KEY: not available ({type(e).__name__}); skipping jev-score")
print("methods:", METHODS)

In [ ]:
!python -m cli run rerank --tag gpu --out /content/out --limit 0 --top-k 20     --methods {" ".join(METHODS)}

In [ ]:
!find /content/out -type f -printf '%-52p %10s bytes
' | sort
!cd /content && zip -qr rerank-gpu.zip out && echo && echo "zipped: /content/rerank-gpu.zip"
from google.colab import files
files.download("/content/rerank-gpu.zip")